In [1]:
from selfmixed_model import *
import torch
import torch.nn as nn
import torch.nn.functional as F

TEST PARTIAL CONVOLUTION:

Sanity checks:

In [2]:
# Define input/output shapes: [B, C, H, W]
in_shape = (1, 3, 64, 64)
out_shape = (1, 8, 64, 64)

# Create an instance of the module
pconv = PartialConv2d(in_shape=in_shape,
                      out_shape=out_shape,
                      kernel_size=3,
                      stride=1,
                      padding_mode='same',
                      bias=True)

# Create a dummy input tensor
x = torch.randn(in_shape)

# Forward pass
output = pconv(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)
print("New mask shape:", pconv.mask.shape)

Input shape: torch.Size([1, 3, 64, 64])
Output shape: torch.Size([1, 8, 64, 64])
New mask shape: torch.Size([1, 8, 64, 64])


In [3]:
# Create an input where some pixels are zeroed out manually
x_test = torch.ones((1, 3, 5, 5))
mask_test = torch.tensor([[[[1.,1.,0.,0.,0.],
                            [1.,1.,0.,0.,0.],
                            [0.,0.,0.,0.,0.],
                            [0.,0.,0.,0.,0.],
                            [0.,0.,0.,0.,0.]]]])

pconv.mask = mask_test.repeat(1, x_test.shape[1], 1, 1) # update internal mask

output_test = pconv(x_test)

print("Current mask: \n", mask_test)
print("Output:\n", output_test)
print("Updated mask:\n", pconv.mask)

Current mask: 
 tensor([[[[1., 1., 0., 0., 0.],
          [1., 1., 0., 0., 0.],
          [0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.]]]])
Output:
 tensor([[[[ -1.2642, -13.7792, -11.8744,   0.0000,   0.0000],
          [-16.9602,  -3.1704,  18.3812,   0.0000,   0.0000],
          [-14.2202,  -1.9529,  17.4464,   0.0000,   0.0000],
          [  0.0000,   0.0000,   0.0000,   0.0000,   0.0000],
          [  0.0000,   0.0000,   0.0000,   0.0000,   0.0000]],

         [[-10.6088,  11.4404,  27.7164,   0.0000,   0.0000],
          [-31.7650,  -7.2528,  -9.6244,   0.0000,   0.0000],
          [-38.6319, -26.5893, -29.3568,   0.0000,   0.0000],
          [  0.0000,   0.0000,   0.0000,   0.0000,   0.0000],
          [  0.0000,   0.0000,   0.0000,   0.0000,   0.0000]],

         [[ -8.7091, -19.4484, -20.0735,   0.0000,   0.0000],
          [ -5.4295, -38.7768, -56.5439,   0.0000,   0.0000],
          [-14.3652, -30.7376, -51.6437,   0.0000,   0.0000],
  

In [4]:
x_test.requires_grad_(True)
output = pconv(x_test)
loss = output.mean()
loss.backward()

print("Gradients exist?", x_test.grad is not None)

Gradients exist? True


Tests:

In [5]:
def test_all_valid_mask():
    in_shape = (1, 3, 16, 16)
    out_shape = (1, 4, 16, 16)
    pconv = PartialConv2d(in_shape=in_shape,
                          out_shape=out_shape,
                          kernel_size=3,
                          padding_mode='same')

    x = torch.randn(in_shape)
    pconv.mask = torch.ones_like(x)

    # Reference conv with same weights
    ref_conv = nn.Conv2d(3, 4, kernel_size=3, padding='same')
    ref_conv.weight.data = pconv.conv.weight.data.clone()
    ref_conv.bias.data = pconv.conv.bias.data.clone()

    y_partial = pconv(x)
    y_ref = ref_conv(x)

    diff = torch.mean(torch.abs(y_partial - y_ref)).item()
    print("Difference between partial and normal conv:", diff)


def test_all_masked_pixels():
    in_shape = (1, 3, 8, 8)
    out_shape = (1, 6, 8, 8)

    pconv = PartialConv2d(in_shape=in_shape,
                          out_shape=out_shape,
                          kernel_size=3,
                          padding_mode='same')

    x = torch.randn(in_shape)
    pconv.mask = torch.zeros_like(x)

    y_partial = pconv(x)

    assert torch.allclose(y_partial.abs(), torch.zeros_like(y_partial), atol=1e-6), \
        "Output should be zero for fully masked input"

    assert pconv.mask.sum() == 0., "New mask should contain only zeros"

    print("All masked pixel test passed!")


def test_mask_broadcasting():
    in_shape = (2, 3, 10, 10)
    out_shape = (2, 5, 10, 10)

    pconv = PartialConv2d(in_shape=in_shape,
                          out_shape=out_shape,
                          kernel_size=3,
                          padding_mode='same',)

    x = torch.randn(in_shape)

    # Forward pass
    y_partial = pconv(x)

    print("Input shape:", x.shape)
    print("Output shape:", y_partial.shape)
    print("Mask shape after forward:", pconv.mask.shape)


def test_gradient_flow():
    in_shape = (1, 3, 8, 8)
    out_shape = (1, 6, 8, 8)

    x = torch.randn(in_shape, requires_grad=True)
    mask = torch.ones_like(x)
    mask[:, :, :4, :] = 0  # half of image masked

    pconv = PartialConv2d(in_shape, out_shape, kernel_size=3,
                          padding_mode='same')

    y = pconv(x)
    loss = y.mean()
    loss.backward()

    grad_mean_valid = x.grad[:, :, 4:, :].abs().mean().item()
    grad_mean_masked = x.grad[:, :, :4, :].abs().mean().item()

    print(
        f"Gradient mean valid={grad_mean_valid:.6f}, masked={grad_mean_masked:.6f}")


def test_stride_dilation_padding():
    in_shape = (1, 3, 32, 32)
    out_shape = (1, 5, 15, 15)  # expected smaller spatial dims due to stride

    pconv = PartialConv2d(in_shape, out_shape, kernel_size=5,
                          stride=2, dilation=2, padding_mode='valid')

    x = torch.randn(in_shape)

    y = pconv(x)

    print("Output shape:", y.shape, "Mask shape:", pconv.mask.shape)

In [6]:
test_all_valid_mask()

Difference between partial and normal conv: 3.7315075397491455


In [7]:
test_all_masked_pixels()

All masked pixel test passed!


In [8]:
test_mask_broadcasting()

Input shape: torch.Size([2, 3, 10, 10])
Output shape: torch.Size([2, 5, 10, 10])
Mask shape after forward: torch.Size([2, 5, 10, 10])


In [9]:
test_gradient_flow()

Gradient mean valid=0.019257, masked=0.018228


In [10]:
test_stride_dilation_padding()

Output shape: torch.Size([1, 5, 12, 12]) Mask shape: torch.Size([1, 5, 12, 12])


TEST ENCODING BLOCK:

In [11]:
def test_encblock_forward():
    in_shape = (1, 3, 64, 64)
    out_shape = (1, 8, 64, 64)

    enc_block = EncBlock(in_shape=in_shape,
                         out_shape=out_shape,
                         mask_ratio=0.5)

    x = torch.randn(in_shape)
    print(x.shape)
    y = enc_block(x)

    print("Input shape:", x.shape)
    print("Output shape:", y.shape)

    # Expected spatial size after AvgPool2d(kernel_size=2,stride=2): half height/width
    assert y.shape[1] == out_shape[1]
    assert y.shape[2] == in_shape[2] // 2
    assert y.shape[3] == in_shape[3] // 2


def test_encblock_mask_behavior():
    in_shape = (1, 3, 32, 32)
    out_shape = (1, 6, 32, 32)

    enc_block = EncBlock(in_shape=in_shape,
                         out_shape=out_shape,
                         mask_ratio=0.7)   # high ratio → more masked pixels

    x = torch.randn(in_shape)

    # Access internal PartialConv layer:
    pconv_layer = enc_block.block[0]

    output = enc_block(x)

    valid_fraction = pconv_layer.mask.mean().item()

    print(f"Fraction of valid pixels in mask: {valid_fraction:.4f}")

    assert valid_fraction <= 1.0 and valid_fraction >= 0.0


def test_encblock_backward():
    in_shape = (1, 3, 32, 32)
    out_shape = (1, 8, 32, 32)

    x = torch.randn(in_shape, requires_grad=True)
    block = EncBlock(in_shape, out_shape)
    y = block(x) 
    y = y.mean()
    y.backward()

    assert x.grad is not None
    print("Gradient mean:", x.grad.abs().mean().item())


def test_encblock_batch_support():
    for batch_size in [1, 4]:
        in_shape = (batch_size, 3, 64, 64)
        out_shape = (batch_size, 8, 64, 64)
        block = EncBlock(in_shape, out_shape)
        x = torch.randn(in_shape)
        y = block(x)

        assert y.shape[0] == batch_size


In [12]:
test_encblock_forward()

torch.Size([1, 3, 64, 64])
Input shape: torch.Size([1, 3, 64, 64])
Output shape: torch.Size([1, 8, 32, 32])


In [13]:
test_encblock_mask_behavior()

Fraction of valid pixels in mask: 1.0000


In [14]:
test_encblock_backward()

Gradient mean: 3.6500179703580216e-05


In [15]:
test_encblock_batch_support()

TEST DECODING BLOCK:

In [16]:
def test_decblock_forward_shape():
    in_channel = 64
    out_channel = 32
    x = torch.randn(1, in_channel, 32, 32)

    dec_block = DecBlock(in_channel=in_channel,
                         out_channel=out_channel,
                         mask_ratio=0.3)

    y = dec_block(x)

    print("Input shape:", x.shape)
    print("Output shape:", y.shape)

    # Expect doubled spatial dimensions due to ConvTranspose2d(stride=2)
    assert y.shape[1] == out_channel
    assert y.shape[2] == x.shape[2] * 2
    assert y.shape[3] == x.shape[3] * 2


# batch_size = [1, 4]
def test_decblock_batch_support():
    for batch_size in [1, 4]:
        in_channel, out_channel = 16, 8
        x = torch.randn(batch_size, in_channel, 16, 16)
        block = DecBlock(in_channel, out_channel)
        y = block(x)

        assert y.shape[0] == batch_size


def test_dropout_effect():
    in_channel, out_channel = 8, 4
    block = DecBlock(in_channel, out_channel,
                     mask_ratio=1)  # high dropout probability

    block.train()
    x = torch.ones((1, in_channel, 8, 8))

    y_train = block(x)

    block.eval()
    y_eval = block(x)

    zeros_train = (y_train == 0).float().mean().item()

    print(f"Fraction of zeros during training={zeros_train:.4f}")

    assert zeros_train > 0


def test_decblock_backward():
    in_ch, out_ch = 16, 8
    block = DecBlock(in_ch, out_ch)

    x = torch.randn((1, in_ch, 32, 32), requires_grad=True)

    loss = block(x).mean()
    loss.backward()

    assert x.grad is not None
    grad_mean = x.grad.abs().mean().item()

    print(f"Mean gradient magnitude={grad_mean:.6f}")


# size = [8 ,16 ,31]
def test_various_input_sizes():
    for size in [8, 16, 31]:
        block = DecBlock(4, 2)
        x = torch.randn((1, 4, size, size))
        y = block(x)

        expected = size*2
        assert list(y.shape) == [1, 2, expected, expected]

In [17]:
test_decblock_forward_shape()

Input shape: torch.Size([1, 64, 32, 32])
Output shape: torch.Size([1, 32, 64, 64])


In [18]:
test_decblock_batch_support()

In [19]:
test_dropout_effect()

Fraction of zeros during training=1.0000


In [20]:
test_decblock_backward()

Mean gradient magnitude=0.000013


In [21]:
test_various_input_sizes()

TEST SELFMIXED MODEL:

In [22]:
def test_calc_dimension_enc_dec():
    in_shape = (1, 3, 128, 128)
    dims_enc = calc_dimension_SM(in_shape, n_layer=5, mode='enc')
    dims_dec = calc_dimension_SM(dims_enc[-1], n_layer=5, mode='dec')

    # Check number of entries
    assert len(dims_enc) == 6
    assert len(dims_dec) == 6

    # Encoding should reduce spatial size progressively
    assert dims_enc[-1][2] < in_shape[2]
    assert dims_enc[-1][3] < in_shape[3]

    # Decoding should increase spatial size again
    assert dims_dec[-1][2] > dims_enc[-1][2]
    assert dims_dec[-1][3] > dims_enc[-1][3]

def test_SMM_forward():
    in_shape = (1, 3, 128, 128)
    out_shape = (1, 3, 128, 128)

    model = SMModel(in_shape=in_shape,
                    out_shape=out_shape,
                    enc_layers=5,
                    dec_layers=5)

    x = torch.randn(in_shape)
    print(model)
    y = model(x)

    print("Input:", x.shape)
    print("Output:", y.shape)

    assert isinstance(y, torch.Tensor)
    assert y.shape[2:] == x.shape[2:], "Output should have same spatial size"


def test_SMM_batch_and_channel_support():
    for batch_size in [1, 4]:
        in_channels = 1
        in_shape = (batch_size, in_channels, 64, 64)
        out_shape = in_shape

        model = SMModel(in_shape, out_shape,
                            enc_layers=5,
                            dec_layers=5)

        x = torch.randn(in_shape)
        y = model(x)

        assert y.shape[0] == batch_size
        assert y.shape[1] == in_channels


def test_SMM_backward_pass():
    in_shape = (1, 3, 64, 64)
    out_shape = in_shape

    model = SMModel(in_shape, out_shape,
                    enc_layers=5,
                    dec_layers=5)

    x = torch.randn(in_shape,
                    requires_grad=True)

    loss = model(x).mean()
    loss.backward()

    assert x.grad is not None
    grad_mean = x.grad.abs().mean().item()

    print(f"Mean gradient magnitude={grad_mean:.6f}")


def test_skip_connections_effect():
    in_shape = (1, 3, 64, 64)
    out_shape = in_shape

    model = SMModel(in_shape, out_shape,
                    enc_layers=5,
                    dec_layers=5)

    x = torch.randn(in_shape)

    with torch.no_grad():
        y = model(x)

       # You could inspect internal tensors manually if needed:
       # For example:
       # s1,s2,o_before,o_after from inside forward() if you instrument it.

    assert torch.isfinite(y).all(), "No NaNs/Infs after skip connections"


def test_training_step():
    in_ch, out_ch, size = (3, 3, 64)

    model = SMModel((1, in_ch, size, size),
                    (1, out_ch, size, size),
                    enc_layers=5,
                    dec_layers=5)

    optimizer = torch.optim.Adam(model.parameters(), lr=.001)

    for epoch in range(2):
        x = torch.randn((4, in_ch, size, size))
        target = torch.randn((4, out_ch, size, size))

        pred = model(x)
        loss = ((pred-target)**2).mean()


        optimizer.zero_grad()

        print("pred.requires_grad:", pred.requires_grad)
        print("loss.requires_grad:", loss.requires_grad)
        loss.backward()
        optimizer.step()

        print(f"Epoch {epoch}, Loss={loss.item():.4f}")

    assert loss.item() > 0

In [23]:
test_calc_dimension_enc_dec()

In [24]:
test_SMM_forward()

SMModel(
  (enc): ModuleList(
    (0): EncBlock(
      (block): Sequential(
        (0): PartialConv2d(
          (conv): Conv2d(3, 24, kernel_size=(3, 3), stride=(1, 1), padding=same)
        )
        (1): BatchNorm2d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): LeakyReLU(negative_slope=0.01)
        (3): AvgPool2d(kernel_size=2, stride=2, padding=0)
      )
    )
    (1-4): 4 x EncBlock(
      (block): Sequential(
        (0): PartialConv2d(
          (conv): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=same)
        )
        (1): BatchNorm2d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): LeakyReLU(negative_slope=0.01)
        (3): AvgPool2d(kernel_size=2, stride=2, padding=0)
      )
    )
  )
  (dec): ModuleList(
    (0): DecBlock(
      (block): Sequential(
        (0): ConvTranspose2d(48, 48, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
        (1): Dropout2d(p=0.5, inplace=False)
        (

In [25]:
test_SMM_batch_and_channel_support()

In [26]:
test_SMM_backward_pass()

Mean gradient magnitude=0.000021


In [27]:
test_skip_connections_effect()

In [28]:
test_training_step()

pred.requires_grad: True
loss.requires_grad: True
Epoch 0, Loss=1.2209
pred.requires_grad: True
loss.requires_grad: True
Epoch 1, Loss=1.1740


In [29]:
x = torch.randn((1,3,64,64), requires_grad=True)
masked_x, m = masks(x)
print("masked_x.requires_grad:", masked_x.requires_grad)

loss = masked_x.mean()
loss.backward()
print("Mean |grad|:", x.grad.abs().mean())

masked_x.requires_grad: True
Mean |grad|: tensor(4.0313e-05)
